In [2]:
# Load and visualize NetCDF slices interactively using parameters read from params.csv
# A. Lefauve, 2026

In [7]:
# STEP 0 of 5 --- Define paths (can be adapted to your local environment) and useful functions
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from types import SimpleNamespace

import utils 
importlib.reload(utils);

# Where is the code is run from and where the params.csv master file is stored
CODE_ROOT = Path("/ccs/home/lefauve/git/INCITE/adrien/")
CSV_PATH = CODE_ROOT / "params.csv"

# Where the data is stored
PROJECT_ROOT = Path("/lustre/orion/cfd135/proj-shared/Hsst")

# Read the params csv file
PARAMS_DF = pd.read_csv(CSV_PATH, dtype={"tStamp": str})
PARAMS_DF.columns = [str(c).strip() for c in PARAMS_DF.columns]

def build_case_from_csv(case_name, tstamp_override=None):
    row = PARAMS_DF.loc[PARAMS_DF["name"].astype(str).str.strip() == str(case_name).strip()]
    if row.empty:
        raise KeyError(f"Unknown case '{case_name}'. Available: {sorted(PARAMS_DF['name'].astype(str).str.strip().tolist())}")
    row = row.iloc[0]

    nx = int(float(row["Nx"]))
    lx = float(row["Lx"])

    p = SimpleNamespace()
    p.name = str(row["name"]).strip()
    p.tStamp = f"{float(tstamp_override):.6f}" if tstamp_override is not None else f"{float(row['tStamp']):.6f}"

    # grid and geometry
    p.Nx = nx
    p.Ny = nx // 2
    p.Nz = nx // 4
    p.Lx = lx
    p.Ly = lx / 2.0
    p.Lz = lx / 4.0

    # directly from CSV
    p.nu = float(row["nu"])
    p.Pr = float(row["Pr"])
    p.Ri = float(row["Ri"])
    p.Reb = float(row["Gn"])
    p.Ek = float(row["Ek"])
    p.Ep = float(row["Ep"])
    p.eps = float(row["ek"])
    p.chi = float(row["ep"])
    p.Gamma1 = float(row["Gamma1"])

    # derived from CSV quantities
    p.N2 = p.Ri
    p.N = float(np.sqrt(p.N2))

    p.dirPath = str(PROJECT_ROOT / p.name / "001_Final") + "/"
    return p


def discover_var_slice_files(slice_dir, case_name):
    """
    Find files of the form:
      <case>_<plane>_<axis><idx>_st<sx>x<sy>_<var>.nc
    Returns nested dict:
      files_by_plane[plane][idx][var] = Path(...)
    """
    patt = re.compile(
        rf"^{re.escape(case_name)}_(xy|xz|yz)_[xyz](\d+)_st(\d+)x(\d+)_([A-Za-z0-9]+)\.nc$"
    )
    files_by_plane = {"xy": {}, "xz": {}, "yz": {}}
    for path in sorted(Path(slice_dir).glob(f"{case_name}_*.nc")):
        m = patt.match(path.name)
        if not m:
            continue
        pl, idx, sx, sy, var = m.groups()
        idx = int(idx)
        files_by_plane.setdefault(pl, {}).setdefault(idx, {})[var] = path
    return files_by_plane


def choose_index(varmap_by_idx, required_vars=("u","v","w","r","ee","chi"), preferred=None):
    if preferred is not None and preferred in varmap_by_idx:
        miss = [v for v in required_vars if v not in varmap_by_idx[preferred]]
        if not miss:
            return preferred
    for idx in sorted(varmap_by_idx):
        miss = [v for v in required_vars if v not in varmap_by_idx[idx]]
        if not miss:
            return idx
    raise RuntimeError("No slice index has all required variables.")

print("Ready to do stuff!")

Ready to do stuff!


In [10]:
# STEP 1 of 5 --- Pick one case and check the available 2D slices and the 3D average quantities from csv file

# CASE = "R1P1"
# CASE = "R1P7"
# CASE = "R1P50"
# CASE = "R4P1"
# CASE = "R4P7"
CASE = "R4P50"
# CASE = "R6P1"
# CASE = "R6P7"
# CASE = "R6P50"
# CASE = "R8P1"
# CASE = "R8P7"
# CASE = "R10P1"
# CASE = "R10P7"

p = build_case_from_csv(CASE)

# where slices were saved
slice_dir = Path(p.dirPath) / "2D_slices"
slice_dir.mkdir(parents=True, exist_ok=True)
print("slice_dir:", slice_dir)

# cache file for the discovered slice filenames
import pickle
cache_path = slice_dir / f"{p.name}_slices_cache.pkl"

if cache_path.exists():
    try:
        with open(cache_path, "rb") as f:
            files_by_plane = pickle.load(f)
        print("Loaded cached slice list:", cache_path)
    except Exception as e:
        print(f"Cache exists but could not be read ({e}), rescanning directory...")
        files_by_plane = discover_var_slice_files(slice_dir, p.name)
        with open(cache_path, "wb") as f:
            pickle.dump(files_by_plane, f)
        print("Saved refreshed cache:", cache_path)
else:
    print("Cache not found, scanning directory...")
    files_by_plane = discover_var_slice_files(slice_dir, p.name)
    with open(cache_path, "wb") as f:
        pickle.dump(files_by_plane, f)
    print("Saved cache:", cache_path)

for pl in ("xy", "xz", "yz"):
    print(pl, "available indices:", sorted(files_by_plane.get(pl, {}).keys())[:20])

# Optional manual overrides; set to None for auto-middle below
PREFERRED_IDX = {
    "xy": None,
    "xz": None,
    "yz": None,
}

# Ideally take the midplanes for the summary plots to come later (only 1 slice for each xy xz yz)
target_idx = {
    "xy": p.Nz // 2,
    "xz": p.Ny // 2,
    "yz": p.Nx // 2,
}

selected_idx = {}
paths = {}

for pl in ("xy", "xz", "yz"):
    if not files_by_plane.get(pl):
        continue

    available = sorted(files_by_plane[pl].keys())

    if PREFERRED_IDX[pl] is not None:
        idx = choose_index(files_by_plane[pl], preferred=PREFERRED_IDX[pl])
    else:
        idx = min(available, key=lambda i: abs(i - target_idx[pl]))

    selected_idx[pl] = idx
    paths[pl] = files_by_plane[pl][idx]

print("target_idx  :", target_idx)
print("selected_idx:", selected_idx)

planes_ok = sorted(paths.keys())
print("planes_ok:", planes_ok)

print("nu =", p.nu)
print("Ri = N2 =", p.N2)
print("Ek =", p.Ek)
print("Ep =", p.Ep)
print("eps =", p.eps)
print("chi =", p.chi)

slice_dir: /lustre/orion/cfd135/proj-shared/Hsst/R4P50/001_Final/2D_slices
Cache not found, scanning directory...
Saved cache: /lustre/orion/cfd135/proj-shared/Hsst/R4P50/001_Final/2D_slices/R4P50_slices_cache.pkl
xy available indices: [667, 1333, 2000, 2667, 3333]
xz available indices: [1333, 2667, 4000, 5333, 6667]
yz available indices: [8000]
target_idx  : {'xy': 2000, 'xz': 4000, 'yz': 8000}
selected_idx: {'xy': 2000, 'xz': 4000, 'yz': 8000}
planes_ok: ['xy', 'xz', 'yz']
nu = 0.000250061
Ri = N2 = 0.15256367
Ek = 0.015789314
Ep = 0.002981864
eps = 0.003144979
chi = 8.19e-05


In [11]:
# STEP 2 of 5 --- Extract slices and check the 2D average from chosen slices is consistent with 3D average from csv file
raw2d = {}
meta2d = {}

required_vars = ("u", "v", "w", "r", "ee", "chi")

for pl in planes_ok:
    raw2d[pl] = {}
    plane_meta = None

    for var in required_vars:
        path = paths[pl][var]
        raw_one, meta_one = utils._read_plane_nc(path)

        if var not in raw_one:
            if len(raw_one) == 1:
                only_key = next(iter(raw_one.keys()))
                raw2d[pl][var] = raw_one[only_key]
            else:
                raise KeyError(f"{path.name}: expected variable '{var}', found {list(raw_one.keys())}")
        else:
            raw2d[pl][var] = raw_one[var]

        plane_meta = meta_one

    meta2d[pl] = plane_meta

    u2 = raw2d[pl]["u"]
    v2 = raw2d[pl]["v"]
    w2 = raw2d[pl]["w"]
    ee2 = raw2d[pl]["ee"]
    chi2 = raw2d[pl]["chi"]

    Ek_slice = float(np.mean(0.5 * (u2**2 + v2**2 + w2**2)))
    ee_slice = float(np.mean(ee2))
    chi_slice = float(np.mean(chi2))

    fixed = {"xy":"iz", "xz":"iy", "yz":"ix"}[pl]
    fixed_val = selected_idx[pl]
    print(
        f"[{pl}] {fixed}={fixed_val}  "
        f"Ek_slice={Ek_slice:.6e}  "
        f"ee_slice={ee_slice:.6e}  chi_slice={chi_slice:.6e}",
        flush=True
    )

stats = {
    "Ek": float(p.Ek),
    "Ep": float(p.Ep),
    "eps_avg": float(p.eps),
    "chi_avg": float(p.chi),
}

print("\n[reference stats from params.csv]")
for k, v in stats.items():
    print(f"  {k}: {v:.6e}")

Ek_slice_mean = float(np.mean([
    np.mean(0.5 * (raw2d[pl]["u"]**2 + raw2d[pl]["v"]**2 + raw2d[pl]["w"]**2))
    for pl in raw2d
]))

Ep_slice_mean = float(np.mean([
    np.mean(1e6*p.Ri/2 * (raw2d[pl]["r"]**2 ))
    for pl in raw2d
]))

eps_slice_mean = float(np.mean([
    float(np.mean(raw2d[pl]["ee"]))
    for pl in raw2d
]))

chi_slice_mean = float(np.mean([
    float(np.mean(raw2d[pl]["chi"]))
    for pl in raw2d
]))

Ek_pct = Ek_slice_mean / p.Ek * 100.0
Ep_pct = Ep_slice_mean / p.Ep * 100.0
eps_pct = eps_slice_mean / p.eps * 100.0
chi_pct = chi_slice_mean / p.chi * 100.0
chiinferred_pct = chi_slice_mean / (p.eps*p.Gamma1) * 100.0

print(f"\nMean 2D slice Ek is {Ek_pct:.2f}% of 3D Ek")
print(f"Mean 2D slice Ep is {Ep_pct:.2f}% of 3D Ep")
print(f"Mean 2D slice eps is {eps_pct:.2f}% of 3D eps")
print(f"Mean 2D slice chi is {chi_pct:.2f}% of 3D chi")
print(f"Mean 2D slice chi is {chiinferred_pct:.2f}% of 3D chi inferred from eps and Gamma1")

[xy] iz=2000  Ek_slice=1.806776e-02  ee_slice=3.936565e-03  chi_slice=5.356948e-04
[xz] iy=4000  Ek_slice=1.599924e-02  ee_slice=1.334360e-02  chi_slice=4.637401e-04
[yz] ix=8000  Ek_slice=1.618136e-02  ee_slice=1.236336e-02  chi_slice=4.460377e-04

[reference stats from params.csv]
  Ek: 1.578931e-02
  Ep: 2.981864e-03
  eps_avg: 3.144979e-03
  chi_avg: 8.190000e-05

Mean 2D slice Ek is 106.08% of 3D Ek
Mean 2D slice Ep is 119.92% of 3D Ep
Mean 2D slice eps is 314.19% of 3D eps
Mean 2D slice chi is 588.31% of 3D chi
Mean 2D slice chi is 99.32% of 3D chi inferred from eps and Gamma1


In [ ]:
# STEP 3 of 5 --- build a SliceBundle 's' containing ONLY derived (normalised/rescaled) fields ready to plot ---
out_vars = {name: utils.VarSlices(name) for name in ["uN", "vN", "wN", "bN", "epslog", "chilog"]}

Ek = float(p.Ek)
Ep = float(p.Ep)
eps_avg = float(p.eps)
#chi_avg = float(p.chi)
chi_avg = float(p.eps*p.Gamma1) # more trustworthy than the actual chi in params (often underestimated)
N2 = float(p.N2)

tiny = 1e-30

for pl in raw2d.keys():
    u2 = raw2d[pl]["u"]
    v2 = raw2d[pl]["v"]
    w2 = raw2d[pl]["w"]
    r2 = raw2d[pl]["r"]
    ee2 = raw2d[pl]["ee"]
    chi2 = raw2d[pl]["chi"]

    out_vars["uN"].set(pl, u2 / np.sqrt(Ek))
    out_vars["vN"].set(pl, v2 / np.sqrt(Ek))
    out_vars["wN"].set(pl, w2 / np.sqrt(Ek))
    out_vars["bN"].set(pl, -1000.0 * p.Ri * r2 / np.sqrt(N2*Ep) )

    with np.errstate(divide="ignore", invalid="ignore"):
        out_vars["epslog"].set(pl, np.log10(np.maximum(ee2, tiny) / eps_avg))
        out_vars["chilog"].set(pl, np.log10(np.maximum(chi2, tiny) / chi_avg))

idx_map = {
    "xy": selected_idx.get("xy", -1),
    "xz": selected_idx.get("xz", -1),
    "yz": selected_idx.get("yz", -1),
}

first_pl = next(iter(meta2d.keys()))
attrs = meta2d[first_pl].get("attrs", {}) if isinstance(meta2d[first_pl], dict) else {}

stride_tuple = (
    int(attrs.get("stride_x", 1)),
    int(attrs.get("stride_y", 1)),
    int(attrs.get("stride_z", 1)),
)

s = utils.SliceBundle(vars=out_vars, idx=idx_map, stride=stride_tuple)

print("planes:", s.available_planes())
print("vars:", s.available_vars())
utils.memory_report(globals(), min_gb=0.01)

In [ ]:
# STEP 4 of 5 --- PLOT A SUMMARY FIGURE FOR EACH SLICE, ALL 6 VARIABLES, AT LOW RESOLUTION (3 FIGURES TOTAL)
planes_avail = tuple(pl for pl in ("xy","xz","yz") if pl in s.available_planes())

if not planes_avail:
    raise RuntimeError("No planes available in SliceBundle 's' (nothing to plot).")

fig_dir = CODE_ROOT / "figures" / p.name
fig_dir.mkdir(parents=True, exist_ok=True)

figs = utils.plot_slices_derived_bundle_multi(
    p, s,
    planes=planes_avail,
    save=True, outdir=str(fig_dir), fmt="png", dpi=300
)


In [9]:
# STEP 5 of 5 --- PLOT FULL RESOLUTION FOR EACH VARIABLE (18 FIGURES TOTAL)
# If you want to plot the other available slices, change the slices to pick in STEP 2

planes_avail = tuple(pl for pl in ("xy", "xz", "yz") if pl in s.available_planes())

if not planes_avail:
    raise RuntimeError("No planes available in SliceBundle 's' (nothing to export).")
    
fig_dir = CODE_ROOT / "figures" / p.name
fig_dir.mkdir(parents=True, exist_ok=True)

paths_exported = []
total = len(planes_avail)

for i, plane in enumerate(planes_avail, 1):
    print(f"[{i}/{total}] Exporting plane: {plane}")

    paths = utils.export_native_resolution_from_bundle_multi(
        p, s,
        planes=[plane],
        outdir=str(fig_dir)
    )

    paths_exported.extend(paths)

# Check memory used by notebook
utils.memory_report(globals(), min_gb=0.01)

[1/3] Exporting plane: xy
[2/3] Exporting plane: xz
[3/3] Exporting plane: yz
u2                          0.07 GB  shape=(6144, 3072)  dtype=float32
v2                          0.07 GB  shape=(6144, 3072)  dtype=float32
w2                          0.07 GB  shape=(6144, 3072)  dtype=float32
ee2                         0.07 GB  shape=(6144, 3072)  dtype=float32
chi2                        0.07 GB  shape=(6144, 3072)  dtype=float32
r2                          0.07 GB  shape=(6144, 3072)  dtype=float32

Notebook memory used     :   9.23 GB (  7.2 %)
Remaining available to me: 118.77 GB ( 92.8 %)
Container memory limit   : 128.00 GB
Total node memory        : 503.30 GB
